# Khmer Emotion Classification with CLAWS, TF–IDF, and Linear SVM

**Course:** ITM 454 002  
**Task:** Seven-class Khmer emotion classification  
**Final model:** CLAWS word tokenization → word TF–IDF → Linear SVM  
**Primary metric:** Macro F1 on a held-out test set

## Project objective

This notebook builds one reproducible statistical NLP pipeline for predicting
`angry`, `fear`, `happy`, `lonely`, `neutral`, `sad`, or `stress` from Khmer text.

The complete experimental policy is:

1. Load and audit all three original Parquet shards.
2. Normalize text and remove verified generator counters.
3. Remove exact duplicates and conflicting-label texts.
4. Collapse near-duplicate generated families and construct the 7,558-row benchmark.
5. Mask direct emotion words **only while detecting stronger template families**.
6. create one template-family-disjoint 80/10/10 train/validation/test split.
7. Train the model on the original, unmasked text.
8. Select the SVM regularization value using validation macro F1.
9. Evaluate the selected model once on the untouched test set.
10. Report performance, errors, limitations, and saved artifacts.

The reported score measures performance on this cleaned, template-heavy benchmark.
It must not be interpreted as accuracy on naturally collected Khmer conversations.


## 1. Environment and reproducibility

In Google Colab, place the three original Parquet shards in
`/content/drive/MyDrive/KhmerEmotionFinal/`. The notebook mounts Drive automatically.
Run the installation cell, restart the runtime only if Colab requests it, and then
choose **Run all**.

Intermediate cleaning results are fingerprinted and cached under
`artifacts_professional/balanced-near-v2/`. The final split, model, figures, and reports
are written to its `final_submission/` subdirectory.


In [ ]:
%pip install -q pandas pyarrow numpy scipy scikit-learn matplotlib seaborn joblib tqdm datasketch claws-tokenizer


In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, asdict
from functools import lru_cache
import hashlib
import importlib.metadata as metadata
import json
import math
import platform
import re
import sqlite3
import sys
import time
import unicodedata
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    pipeline_version: str = "balanced-near-v2"
    near_duplicate_threshold: float = 0.65
    shingle_size: int = 5
    minhash_permutations: int = 128
    template_similarity: float = 0.80
    template_neighbors: int = 6
    svm_c_values: tuple = (0.1, 1.0, 10.0)
    token_checkpoint_every: int = 250
    bootstrap_repeats: int = 2_000
    force_rebuild_dedup: bool = False
    force_rebuild_near_duplicates: bool = False
    force_retokenize: bool = False

CFG = ExperimentConfig()
np.random.seed(CFG.seed)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/KhmerEmotionFinal")
else:
    ROOT = Path.cwd()

if not ROOT.is_dir():
    raise FileNotFoundError(f"Project directory does not exist: {ROOT}")

FILES = [ROOT / f"train-{index:05d}-of-00003.parquet" for index in range(3)]
missing_files = [str(path) for path in FILES if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing original Parquet shards: {missing_files}")

LABELS = ["angry", "fear", "happy", "lonely", "neutral", "sad", "stress"]
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

OUT = ROOT / "artifacts_professional" / CFG.pipeline_version
FINAL_OUT = OUT / "final_submission"
OUT.mkdir(parents=True, exist_ok=True)
FINAL_OUT.mkdir(parents=True, exist_ok=True)

packages = [
    "pandas", "pyarrow", "numpy", "scipy", "scikit-learn",
    "claws-tokenizer", "datasketch", "joblib",
]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"

print("Python:", platform.python_version())
print("Project directory:", ROOT)
print("Final output directory:", FINAL_OUT)
print("Configuration:", json.dumps(asdict(CFG), indent=2))
print("Package versions:", versions)


## 2. Input fingerprint and cache identity

A cache must be invalidated when either the input data **or the cleaning policy** changes. File modification time alone is not sufficient. The manifest therefore records SHA-256 hashes, schema, configuration, and the explicit pipeline version.

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "pipeline_version": CFG.pipeline_version,
    "labels": LABELS,
    "files": [
        {
            "file": path.name,
            "rows": pq.ParquetFile(path).metadata.num_rows,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "schema": str(pq.ParquetFile(path).schema_arrow),
        }
        for path in FILES
    ],
}
manifest_fingerprint = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()

display(pd.DataFrame(manifest["files"])[["file", "rows", "bytes", "sha256"]])
print("Input/pipeline fingerprint:", manifest_fingerprint)
(OUT / "input_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

## 3. Cleaning policy and exact deduplication

The pipeline preserves raw examples for auditing but models only cleaned text.

Cleaning steps:

1. Unicode NFC normalization.
2. Explicit invisible/control separators converted to spaces.
3. Whitespace collapsed.
4. Verified dataset generator suffixes removed:
   - bare counters such as `(1)`;
   - `(ករណីសិក្សាទី N)`;
   - `(ករណីទី N)`;
   - `(ការគិតអប់រំចិត្តទី N)`.
5. A whitespace-insensitive exact-deduplication key is created.
6. Exact texts with conflicting labels are excluded.

In [ ]:
SPACE_RE = re.compile(r"\s+")
INVISIBLE_SEPARATORS_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
KHMER_RE = re.compile(r"[\u1780-\u17FF]")
BARE_COUNTER_RE = re.compile(
    r"[\s\u200b\u200c\u200d\ufeff]*\(([0-9០-៩]+)\)"
    r"[\s\u200b\u200c\u200d\ufeff]*$"
)
GENERATOR_MARKER_RE = re.compile(
    r"[\s\u200b\u200c\u200d\ufeff]*"
    r"\((?:ការគិតអប់រំចិត្តទី|ករណីសិក្សាទី|ករណីទី)\s*[0-9០-៩]+\)"
    r"[\s\u200b\u200c\u200d\ufeff]*$"
)

@lru_cache(maxsize=200_000)
def normalize_base(value):
    value = unicodedata.normalize("NFC", value)
    value = INVISIBLE_SEPARATORS_RE.sub(" ", value)
    value = "".join(" " if unicodedata.category(ch) == "Cc" else ch for ch in value)
    return SPACE_RE.sub(" ", value).strip()

def remove_generated_suffix(value):
    marker = GENERATOR_MARKER_RE.search(value)
    if marker:
        return GENERATOR_MARKER_RE.sub("", value).strip(), marker.group(0).strip(), "generator_marker"
    counter = BARE_COUNTER_RE.search(value)
    if counter:
        return BARE_COUNTER_RE.sub("", value).strip(), counter.group(0).strip(), "bare_counter"
    return value, None, None

def canonical_key(value):
    return SPACE_RE.sub("", value)

def prepare_text(raw_text):
    base = normalize_base(raw_text)
    model_text, removed_suffix, removal_rule = remove_generated_suffix(base)
    return {
        "base_text": base,
        "model_text": model_text,
        "pre_suffix_key": canonical_key(base),
        "dedup_key": canonical_key(model_text),
        "removed_suffix": removed_suffix,
        "removal_rule": removal_rule,
    }

examples = [
    "ខ្ញុំសប្បាយចិត្តណាស់។ (1)",
    "ខ្ញុំមានអារម្មណ៍សោកសៅ។ (ករណីសិក្សាទី 10)",
    "ខ្ញុំមានអារម្មណ៍ខឹង។ (ការគិតអប់រំចិត្តទី 66972)",
]
display(pd.DataFrame([{"raw": text, **prepare_text(text)} for text in examples]))

### Build or reuse the audited deduplication database

This stage scans all shards in bounded batches. It reports missing/invalid rows, unique keys before suffix removal, unique keys after suffix removal, duplicate excess, label conflicts, and per-label unique counts.

The validated cache is reused automatically. Set `CFG.force_rebuild_dedup=True` only when intentionally rebuilding this stage.

In [ ]:
DEDUP_WORK_DIR = Path("/content/khmer_emotion_cache") if IN_COLAB else OUT
DEDUP_WORK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DEDUP_WORK_DIR / "deduplicated.sqlite"
AUDIT_PATH = OUT / "data_audit.json"
CACHE_META_PATH = OUT / "dedup_cache_meta.json"
expected_cache_meta = {
    "manifest_fingerprint": manifest_fingerprint,
    "pipeline_version": CFG.pipeline_version,
}
cache_valid = (
    DB_PATH.exists()
    and AUDIT_PATH.exists()
    and CACHE_META_PATH.exists()
    and json.loads(CACHE_META_PATH.read_text(encoding="utf-8")) == expected_cache_meta
)

if CFG.force_rebuild_dedup or not cache_valid:
    for suffix in ("", "-wal", "-shm"):
        candidate = Path(str(DB_PATH) + suffix)
        if candidate.exists():
            candidate.unlink()

    db = sqlite3.connect(DB_PATH)
    db.execute("PRAGMA journal_mode=WAL")
    db.execute("PRAGMA synchronous=NORMAL")
    db.execute("""
        CREATE TABLE texts (
            dedup_key TEXT PRIMARY KEY,
            model_text TEXT NOT NULL,
            raw_example TEXT NOT NULL,
            label_mask INTEGER NOT NULL,
            copies INTEGER NOT NULL,
            suffix_copies INTEGER NOT NULL,
            first_id TEXT NOT NULL,
            first_file TEXT NOT NULL
        )
    """)
    db.execute("CREATE TABLE pre_suffix_keys (key TEXT PRIMARY KEY)")
    upsert = """
        INSERT INTO texts VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(dedup_key) DO UPDATE SET
            label_mask = texts.label_mask | excluded.label_mask,
            copies = texts.copies + excluded.copies,
            suffix_copies = texts.suffix_copies + excluded.suffix_copies
    """

    rejected = Counter()
    raw_labels = Counter()
    raw_rows = 0
    rows_with_generated_suffix = 0
    generated_suffix_examples = []
    started = time.perf_counter()

    for path in FILES:
        parquet_file = pq.ParquetFile(path)
        progress = tqdm(
            parquet_file.iter_batches(batch_size=50_000, columns=["id", "text", "label"]),
            total=math.ceil(parquet_file.metadata.num_rows / 50_000),
            desc=f"Audit {path.name}",
        )
        for batch in progress:
            rows = batch.to_pydict()
            grouped = {}
            pre_suffix_batch = set()
            for ident, raw_text, label in zip(rows["id"], rows["text"], rows["label"]):
                raw_rows += 1
                label = label.strip().lower() if isinstance(label, str) else "<missing>"
                raw_labels[label] += 1

                if ident is None:
                    rejected["missing_id"] += 1
                    continue
                if not isinstance(raw_text, str):
                    rejected["missing_or_nonstring_text"] += 1
                    continue
                if label not in label2id:
                    rejected["unexpected_label"] += 1
                    continue

                prepared = prepare_text(raw_text)
                model_text = prepared["model_text"]
                if not model_text:
                    rejected["empty_after_cleaning"] += 1
                    continue
                if not KHMER_RE.search(model_text):
                    rejected["no_khmer_character"] += 1
                    continue

                had_generated_suffix = prepared["removed_suffix"] is not None
                rows_with_generated_suffix += int(had_generated_suffix)
                if had_generated_suffix and len(generated_suffix_examples) < 20:
                    generated_suffix_examples.append({
                        "raw_text": raw_text,
                        "model_text": model_text,
                        "removed_suffix": prepared["removed_suffix"],
                        "label": label,
                    })

                pre_suffix_batch.add(prepared["pre_suffix_key"])
                key = prepared["dedup_key"]
                mask = 1 << label2id[label]
                if key in grouped:
                    grouped[key][3] |= mask
                    grouped[key][4] += 1
                    grouped[key][5] += int(had_generated_suffix)
                else:
                    grouped[key] = [
                        key, model_text, raw_text, mask, 1, int(had_generated_suffix),
                        str(ident), path.name,
                    ]

            db.executemany("INSERT OR IGNORE INTO pre_suffix_keys VALUES (?)",
                           [(key,) for key in pre_suffix_batch])
            db.executemany(upsert, grouped.values())
            db.commit()

    unique_before_suffix_removal = db.execute("SELECT COUNT(*) FROM pre_suffix_keys").fetchone()[0]
    unique_keys, valid_rows = db.execute(
        "SELECT COUNT(*), COALESCE(SUM(copies), 0) FROM texts"
    ).fetchone()
    conflict_keys, conflict_rows = db.execute(
        "SELECT COUNT(*), COALESCE(SUM(copies), 0) FROM texts "
        "WHERE (label_mask & (label_mask - 1)) != 0"
    ).fetchone()

    eligible_by_label = {}
    for i, label in id2label.items():
        eligible_by_label[label] = db.execute(
            "SELECT COUNT(*) FROM texts WHERE label_mask = ?", (1 << i,)
        ).fetchone()[0]

    audit = {
        "raw_rows": raw_rows,
        "rejected": dict(rejected),
        "valid_rows_before_dedup": valid_rows,
        "unique_keys_before_suffix_removal": unique_before_suffix_removal,
        "unique_keys_after_suffix_removal": unique_keys,
        "unique_keys_collapsed_by_suffix_rules": unique_before_suffix_removal - unique_keys,
        "rows_with_generated_suffix": rows_with_generated_suffix,
        "duplicate_or_artifact_excess_rows": valid_rows - unique_keys,
        "conflicting_keys_excluded": conflict_keys,
        "rows_in_conflicting_keys": conflict_rows,
        "eligible_unique_texts": unique_keys - conflict_keys,
        "raw_label_counts": dict(raw_labels),
        "eligible_unique_by_label": eligible_by_label,
        "generated_suffix_examples": generated_suffix_examples,
        "runtime_minutes": (time.perf_counter() - started) / 60,
    }
    assert raw_rows == valid_rows + sum(rejected.values())

    AUDIT_PATH.write_text(
        json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    CACHE_META_PATH.write_text(
        json.dumps(expected_cache_meta, indent=2), encoding="utf-8"
    )
    db.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    db.close()
else:
    audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
    print("Reused validated deduplication cache.")

db = sqlite3.connect(DB_PATH)
print(json.dumps({k: v for k, v in audit.items()
                  if k not in {"generated_suffix_examples"}}, ensure_ascii=False, indent=2))
display(pd.DataFrame(audit["generated_suffix_examples"]).head(10))

### Audit summary and decision gate

Do not continue until the removed-suffix examples, label counts, and exact-deduplication totals are plausible. These are dataset findings that must appear in the final report.

In [ ]:
audit_table = pd.DataFrame({
    "raw_rows": pd.Series(audit["raw_label_counts"]),
    "eligible_unique_texts": pd.Series(audit["eligible_unique_by_label"]),
}).reindex(LABELS)
audit_table["duplicate_or_removed_fraction"] = (
    1 - audit_table["eligible_unique_texts"] / audit_table["raw_rows"]
)
display(audit_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
audit_table[["raw_rows", "eligible_unique_texts"]].plot.bar(
    ax=axes[0], logy=True, title="Raw rows vs. eligible unique texts (log scale)"
)
audit_table["duplicate_or_removed_fraction"].plot.bar(
    ax=axes[1], color="#b54a4a", title="Duplicate/artifact fraction by label"
)
axes[1].set_ylim(0, 1)
axes[0].set_ylabel("count")
plt.tight_layout()

if audit["conflicting_keys_excluded"]:
    warnings.warn(
        f'{audit["conflicting_keys_excluded"]:,} exact-text keys have conflicting labels '
        "and will be excluded."
    )
assert (audit_table["eligible_unique_texts"] > 0).all(), (
    "At least one label has no eligible unique text after cleaning."
)

## 4. Load every eligible exact-unique text

No per-class cap is applied here. Every valid exact-unique text is loaded before near-duplicate removal. This cell reports the natural cleaned class distribution.

In [ ]:
records = []
query = """
    SELECT dedup_key, model_text, raw_example, label_mask, copies,
           suffix_copies, first_id, first_file
    FROM texts
    WHERE (label_mask & (label_mask - 1)) = 0
    ORDER BY dedup_key
"""
for key, text, raw_example, mask, copies, suffix_copies, source_id, source_file in db.execute(query):
    label = id2label[int(mask).bit_length() - 1]
    records.append({
        "text_key": key,
        "text": text,
        "raw_example": raw_example,
        "label": label,
        "original_copies": copies,
        "suffix_copies": suffix_copies,
        "source_id": source_id,
        "source_file": source_file,
    })
corpus = pd.DataFrame(records)
db.close()

assert corpus["text_key"].is_unique
assert set(corpus["label"]) == set(LABELS)
corpus_identity = {
    "pipeline_version": CFG.pipeline_version,
    "seed": CFG.seed,
    "rows": len(corpus),
}
corpus_fingerprint = hashlib.sha256(
    ("|".join(corpus["text_key"]) + "|" + json.dumps(corpus_identity, sort_keys=True))
    .encode("utf-8")
).hexdigest()
display(corpus.groupby("label").agg(
    exact_unique_texts=("text", "size"),
    median_characters=("text", lambda values: int(values.str.len().median())),
    max_original_copies=("original_copies", "max"),
))
print("Exact-unique corpus:", len(corpus))
print("Corpus fingerprint:", corpus_fingerprint)

## 5. Near-duplicate removal and class balancing

Exact deduplication cannot detect lightly rewritten generated templates. Within each label, MinHash proposes similar character-5-gram candidates; exact Jaccard similarity verifies every proposed merge. Connected components define near-duplicate families.

One deterministic representative is retained from each family. All near-unique examples from the six non-stress labels are retained. Only the oversized `stress` label is deterministically capped at the size of the largest non-stress label. No artificial duplicate is created.

This stage can take time on the full corpus. Its result is cached by corpus fingerprint and configuration.

In [ ]:
from datasketch import MinHash, MinHashLSH

NEAR_GROUP_PATH = OUT / "near_duplicate_groups.parquet"
NEAR_META_PATH = OUT / "near_duplicate_meta.json"
near_meta = {
    "corpus_fingerprint": corpus_fingerprint,
    "threshold": CFG.near_duplicate_threshold,
    "shingle_size": CFG.shingle_size,
    "minhash_permutations": CFG.minhash_permutations,
}
near_cache_valid = (
    NEAR_GROUP_PATH.exists()
    and NEAR_META_PATH.exists()
    and json.loads(NEAR_META_PATH.read_text(encoding="utf-8")) == near_meta
)

NUMBER_RE = re.compile(r"[0-9០-៩]+")
def character_shingles(text, size=CFG.shingle_size):
    normalized = canonical_key(NUMBER_RE.sub("<NUM>", text))
    if len(normalized) <= size:
        return {normalized}
    return {normalized[i:i + size] for i in range(len(normalized) - size + 1)}

if near_cache_valid and not CFG.force_rebuild_near_duplicates:
    near_groups = pd.read_parquet(NEAR_GROUP_PATH)
    corpus = corpus.drop(columns=["near_duplicate_group"], errors="ignore").merge(
        near_groups, on="text_key", how="left", validate="one_to_one"
    )
    print("Reused validated near-duplicate groups.")
else:
    parent = list(range(len(corpus)))
    rank = [0] * len(corpus)

    def find(item):
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    def union(a, b):
        root_a, root_b = find(a), find(b)
        if root_a == root_b:
            return
        if rank[root_a] < rank[root_b]:
            root_a, root_b = root_b, root_a
        parent[root_b] = root_a
        if rank[root_a] == rank[root_b]:
            rank[root_a] += 1

    candidate_pairs = 0
    verified_pairs = 0
    for label in LABELS:
        label_positions = np.flatnonzero(corpus["label"].to_numpy() == label)
        lsh = MinHashLSH(
            threshold=CFG.near_duplicate_threshold,
            num_perm=CFG.minhash_permutations,
        )
        for position in tqdm(label_positions, desc=f"Near duplicates: {label}"):
            shingles = character_shingles(corpus.iloc[position]["text"])
            signature = MinHash(num_perm=CFG.minhash_permutations, seed=CFG.seed)
            for shingle in sorted(shingles):
                signature.update(shingle.encode("utf-8"))
            for candidate_name in lsh.query(signature):
                candidate = int(candidate_name)
                candidate_pairs += 1
                other = character_shingles(corpus.iloc[candidate]["text"])
                similarity = len(shingles & other) / len(shingles | other)
                if similarity >= CFG.near_duplicate_threshold:
                    union(position, candidate)
                    verified_pairs += 1
            lsh.insert(str(position), signature)

    components = {}
    for position in range(len(corpus)):
        components.setdefault(find(position), []).append(position)
    group_ids = {}
    for members in components.values():
        stable_keys = sorted(corpus.iloc[members]["text_key"])
        group_id = hashlib.sha1("|".join(stable_keys).encode("utf-8")).hexdigest()
        for member in members:
            group_ids[member] = group_id
    corpus["near_duplicate_group"] = [
        group_ids[position] for position in range(len(corpus))
    ]
    corpus[["text_key", "near_duplicate_group"]].to_parquet(
        NEAR_GROUP_PATH, index=False
    )
    NEAR_META_PATH.write_text(json.dumps(near_meta, indent=2), encoding="utf-8")
    print(f"Candidate pairs: {candidate_pairs:,}; verified pairs: {verified_pairs:,}")

def stable_priority(key, seed=CFG.seed):
    return int.from_bytes(
        hashlib.blake2b(f"{seed}|{key}".encode("utf-8"), digest_size=8).digest(),
        "big",
    )

corpus["selection_priority"] = corpus["text_key"].map(stable_priority)
near_unique = (
    corpus.sort_values(["near_duplicate_group", "selection_priority"])
    .drop_duplicates("near_duplicate_group", keep="first")
    .copy()
)
near_counts = near_unique["label"].value_counts().reindex(LABELS)
NON_STRESS_LABELS = [label for label in LABELS if label != "stress"]
STRESS_CAP = int(near_counts.loc[NON_STRESS_LABELS].max())

retained_parts = []
for label in LABELS:
    label_rows = near_unique.loc[near_unique["label"] == label].sort_values(
        "selection_priority"
    )
    if label == "stress":
        label_rows = label_rows.head(STRESS_CAP)
    retained_parts.append(label_rows)
sample = (
    pd.concat(retained_parts, ignore_index=True)
    .sort_values(["label", "selection_priority"])
    .reset_index(drop=True)
)
retained_counts = sample["label"].value_counts().reindex(LABELS)
assert sample["text_key"].is_unique
assert sample["near_duplicate_group"].is_unique
assert retained_counts["stress"] <= STRESS_CAP
assert retained_counts.loc[NON_STRESS_LABELS].equals(
    near_counts.loc[NON_STRESS_LABELS]
)

sample_identity = {
    "pipeline_version": CFG.pipeline_version,
    "corpus_fingerprint": corpus_fingerprint,
    "near_duplicate_threshold": CFG.near_duplicate_threshold,
    "stress_cap": STRESS_CAP,
    "retained_counts": retained_counts.to_dict(),
}
sample_fingerprint = hashlib.sha256(
    ("|".join(sample["text_key"]) + "|" + json.dumps(sample_identity, sort_keys=True))
    .encode("utf-8")
).hexdigest()

near_unique.to_parquet(OUT / "near_unique_corpus.parquet", index=False)
sample.to_parquet(OUT / "balanced_benchmark.parquet", index=False)
print("Exact-unique rows:", len(corpus))
print("Near-unique rows:", len(near_unique))
print("Near-unique counts:", near_counts.to_dict())
print("Stress cap:", STRESS_CAP)
print("Retained counts:", retained_counts.to_dict())
print("Final approximately balanced benchmark:", len(sample))
print("Benchmark fingerprint:", sample_fingerprint)

### Near-duplicate and balance audit

Inspect the largest families before accepting the benchmark. The report must state the exact threshold, texts removed as near duplicates, per-label counts before capping, the stress cap, and final retained counts.

In [ ]:
group_summary = (
    corpus.groupby("near_duplicate_group")
    .agg(size=("text_key", "size"), label=("label", "first"))
    .sort_values("size", ascending=False)
)
print(
    f"Near-duplicate families: {len(group_summary):,}; "
    f"singletons: {group_summary['size'].eq(1).sum():,}; "
    f"largest family: {group_summary['size'].max():,}"
)
display(group_summary["size"].value_counts().sort_index().head(25).rename(
    "number_of_families"
).to_frame())
largest = group_summary.head(10).index
display(
    corpus.loc[corpus["near_duplicate_group"].isin(largest),
               ["near_duplicate_group", "label", "text"]]
    .sort_values(["near_duplicate_group", "text"])
    .head(60)
)
balance_audit = pd.DataFrame({
    "exact_unique": corpus["label"].value_counts(),
    "near_unique": near_counts,
    "retained_for_modeling": retained_counts,
}).reindex(LABELS)
balance_audit["removed_near_duplicate"] = (
    balance_audit["exact_unique"] - balance_audit["near_unique"]
)
balance_audit["removed_for_balance"] = (
    balance_audit["near_unique"] - balance_audit["retained_for_modeling"]
)
display(balance_audit)
balance_audit.to_csv(OUT / "balance_audit.csv")

## 6. Stronger template families and the final split

The earlier MinHash stage removes close rewrites from the full corpus and retains one
representative per family. A second, stricter audit is now performed on the 7,558
retained examples to prevent broader writing templates from crossing partitions.

Direct emotion cues are masked only in a temporary copy used for template discovery.
Character 3–5 gram TF–IDF and cosine similarity connect highly similar templates.
Connected components form template families, and `StratifiedGroupKFold` assigns every
complete family to exactly one partition.

**The original unmasked `text` column is used for tokenization, training, testing, and
inference.**


In [ ]:
assert len(sample) == 7_558
assert sample["text_key"].is_unique
assert set(sample["label"]) == set(LABELS)

print("Final benchmark rows:", len(sample))
display(sample["label"].value_counts().reindex(LABELS).rename("rows").to_frame())
display(
    sample.groupby("label", group_keys=False)
    .sample(n=2, random_state=CFG.seed)[["text", "label"]]
)

DIRECT_CUES = {
    "angry": ["ខឹង", "កំហឹង", "ក្រហាយ"],
    "fear": ["ភ័យខ្លាច", "ខ្លាច", "ភ័យ"],
    "happy": ["សប្បាយចិត្ត", "រីករាយ", "ស្រស់ស្រាយ"],
    "lonely": ["ម្នាក់ឯង", "ឯកោ", "ឯកា"],
    "neutral": ["ស្ងប់ស្ងាត់", "ធម្មតា"],
    "sad": ["សោកសៅ", "ខូចចិត្ត", "ទឹកភ្នែក"],
    "stress": ["សម្ពាធ", "តានតឹង", "ស្ត្រេស", "ថប់"],
}
ALL_DIRECT_CUES = sorted(
    {cue for cues in DIRECT_CUES.values() for cue in cues},
    key=len,
    reverse=True,
)
GROUP_MASK = "[អារម្មណ៍]"

def mask_for_template_discovery(text):
    masked = text
    for cue in ALL_DIRECT_CUES:
        masked = masked.replace(cue, GROUP_MASK)
    return masked

sample["grouping_text"] = sample["text"].map(mask_for_template_discovery)
sample["cue_was_masked"] = sample["grouping_text"].ne(sample["text"])
mask_audit = (
    sample.groupby("label")["cue_was_masked"]
    .agg(masked_rows="sum", total_rows="count", masked_fraction="mean")
    .reindex(LABELS)
)
display(mask_audit)

template_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.995,
    max_features=250_000,
    sublinear_tf=True,
    dtype=np.float32,
)
template_vectors = template_vectorizer.fit_transform(sample["grouping_text"])
neighbor_count = min(CFG.template_neighbors, len(sample))
template_neighbors = NearestNeighbors(
    n_neighbors=neighbor_count,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
).fit(template_vectors)
template_distances, template_indices = template_neighbors.kneighbors(template_vectors)

edge_rows, edge_columns = [], []
for row_index in range(len(sample)):
    for distance, neighbor_index in zip(
        template_distances[row_index, 1:], template_indices[row_index, 1:]
    ):
        similarity = 1.0 - float(distance)
        if similarity >= CFG.template_similarity:
            edge_rows.extend([row_index, int(neighbor_index)])
            edge_columns.extend([int(neighbor_index), row_index])

template_graph = coo_matrix(
    (np.ones(len(edge_rows), dtype=np.uint8), (edge_rows, edge_columns)),
    shape=(len(sample), len(sample)),
).tocsr()
template_group_count, template_groups = connected_components(
    template_graph, directed=False
)
sample["template_group"] = template_groups

template_audit = (
    sample.groupby("template_group")
    .agg(size=("text_key", "size"), labels=("label", "nunique"))
    .sort_values("size", ascending=False)
)
print("Stronger template families:", template_group_count)
print(
    "Rows in non-singleton families:",
    int(template_audit.loc[template_audit["size"] > 1, "size"].sum()),
)
print("Largest stronger family:", int(template_audit["size"].max()))
print("Families containing multiple labels:", int(template_audit["labels"].gt(1).sum()))
display(template_audit.head(20))

fold_assignment = np.full(len(sample), -1, dtype=int)
fold_builder = StratifiedGroupKFold(
    n_splits=10, shuffle=True, random_state=CFG.seed
)
for fold_number, (_, fold_indices) in enumerate(
    fold_builder.split(
        sample["grouping_text"],
        sample["label"],
        groups=sample["template_group"],
    )
):
    fold_assignment[fold_indices] = fold_number

assert (fold_assignment >= 0).all()
sample["split"] = np.where(
    fold_assignment == 0,
    "test",
    np.where(fold_assignment == 1, "validation", "train"),
)

assert sample.groupby("template_group")["split"].nunique().max() == 1
split_groups = {
    name: set(sample.loc[sample["split"] == name, "template_group"])
    for name in ["train", "validation", "test"]
}
assert split_groups["train"].isdisjoint(split_groups["validation"])
assert split_groups["train"].isdisjoint(split_groups["test"])
assert split_groups["validation"].isdisjoint(split_groups["test"])

split_table = pd.crosstab(sample["split"], sample["label"]).reindex(
    index=["train", "validation", "test"], columns=LABELS, fill_value=0
)
display(split_table)
print("Split totals:", split_table.sum(axis=1).to_dict())
print("Verified: no stronger template family crosses partitions.")

split_columns = [
    "text_key", "text", "label", "near_duplicate_group",
    "template_group", "split", "source_id", "source_file",
    "original_copies", "suffix_copies",
]
sample[split_columns].to_parquet(
    FINAL_OUT / "final_template_group_split.parquet", index=False
)
mask_audit.to_csv(FINAL_OUT / "template_mask_audit.csv")
template_audit.to_csv(FINAL_OUT / "template_family_audit.csv")

split_table.plot.bar(figsize=(11, 5))
plt.title("Class distribution in the final template-family-disjoint split")
plt.xlabel("Partition")
plt.ylabel("Examples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FINAL_OUT / "final_split_distribution.png", dpi=180, bbox_inches="tight")
plt.show()


## 7. CLAWS tokenization

CLAWS performs Khmer word segmentation for the statistical model. Tokenization is
cached by benchmark fingerprint and CLAWS version so an interrupted run can resume.
The cache is also checked against the SHA-256 hash of every original text.


In [ ]:
import builtins

standard_open = builtins.open

def open_claws_vocabulary_utf8(file, mode="r", *args, **kwargs):
    is_claws_vocabulary = (
        str(file).replace("\\", "/").endswith("claws_tokenizer/models/vocab.json")
        and "b" not in mode
    )
    if is_claws_vocabulary:
        kwargs.setdefault("encoding", "utf-8")
    return standard_open(file, mode, *args, **kwargs)

builtins.open = open_claws_vocabulary_utf8
try:
    from claws_tokenizer import segment
finally:
    builtins.open = standard_open

claws_version = metadata.version("claws-tokenizer")
TOKEN_DB_PATH = OUT / "claws_tokens.sqlite"
TOKEN_META_PATH = OUT / "claws_token_cache_meta.json"
token_meta = {
    "sample_fingerprint": sample_fingerprint,
    "pipeline_version": CFG.pipeline_version,
    "claws_version": claws_version,
}
token_cache_valid = (
    TOKEN_DB_PATH.exists()
    and TOKEN_META_PATH.exists()
    and json.loads(TOKEN_META_PATH.read_text(encoding="utf-8")) == token_meta
)

if CFG.force_retokenize or not token_cache_valid:
    for suffix in ("", "-wal", "-shm"):
        candidate = Path(str(TOKEN_DB_PATH) + suffix)
        if candidate.exists():
            candidate.unlink()
    TOKEN_META_PATH.write_text(json.dumps(token_meta, indent=2), encoding="utf-8")

token_db = sqlite3.connect(TOKEN_DB_PATH)
token_db.execute("PRAGMA journal_mode=WAL")
token_db.execute(
    "CREATE TABLE IF NOT EXISTS tokens ("
    "text_key TEXT PRIMARY KEY, "
    "text_sha256 TEXT NOT NULL, "
    "tokens_json TEXT NOT NULL)"
)
existing_keys = {row[0] for row in token_db.execute("SELECT text_key FROM tokens")}
missing_tokens = sample.loc[
    ~sample["text_key"].isin(existing_keys), ["text_key", "text"]
]
print(
    f"CLAWS {claws_version}: {len(existing_keys & set(sample.text_key)):,} cached; "
    f"{len(missing_tokens):,} remaining."
)

completed_now = 0
try:
    for row in tqdm(
        missing_tokens.itertuples(index=False),
        total=len(missing_tokens),
        desc="CLAWS tokenization",
    ):
        tokens = [token.strip() for token in segment(row.text) if token.strip()]
        if not tokens:
            raise RuntimeError(f"CLAWS returned no tokens for key {row.text_key}")
        text_hash = hashlib.sha256(row.text.encode("utf-8")).hexdigest()
        token_db.execute(
            "INSERT OR REPLACE INTO tokens VALUES (?, ?, ?)",
            (row.text_key, text_hash, json.dumps(tokens, ensure_ascii=False)),
        )
        completed_now += 1
        if completed_now % CFG.token_checkpoint_every == 0:
            token_db.commit()
finally:
    token_db.commit()

token_rows = pd.read_sql_query(
    "SELECT text_key, text_sha256, tokens_json FROM tokens", token_db
)
token_db.execute("PRAGMA wal_checkpoint(TRUNCATE)")
token_db.close()

sample["row_order"] = np.arange(len(sample))
sample = sample.drop(
    columns=["tokens", "tokenized_text", "tokens_json", "text_sha256"],
    errors="ignore",
).merge(token_rows, on="text_key", how="left", validate="one_to_one", sort=False)
sample = sample.sort_values("row_order").drop(columns="row_order").reset_index(drop=True)

if sample["tokens_json"].isna().any():
    raise RuntimeError(f"{int(sample['tokens_json'].isna().sum())} texts were not tokenized")

expected_hashes = sample["text"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)
assert expected_hashes.equals(sample["text_sha256"]), "Stale token cache detected"

sample["tokens"] = sample["tokens_json"].map(json.loads)
sample["tokenized_text"] = sample["tokens"].map(" ".join)
sample = sample.drop(columns=["tokens_json", "text_sha256"])

display(
    sample[["text", "tokenized_text", "label"]]
    .sample(min(15, len(sample)), random_state=CFG.seed)
)
print(
    "Token-count percentiles:",
    sample["tokens"].map(len).quantile([0, .25, .5, .75, .9, .95, .99, 1]).to_dict(),
)


## 8. Training and validation-based model selection

TF–IDF is fitted on the training partition only. The validation partition selects the
SVM `C` value using macro F1, which gives equal importance to all seven classes. The
test partition is not inspected during feature fitting or model selection.


In [ ]:
def metric_row(y_true, y_pred):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": macro_f1,
        "f1_weighted": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="weighted",
            zero_division=0,
        ),
    }

parts = {
    name: sample.loc[sample["split"] == name].copy().reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=300_000,
    sublinear_tf=True,
    dtype=np.float32,
)
x_train = vectorizer.fit_transform(parts["train"]["tokenized_text"])
x_validation = vectorizer.transform(parts["validation"]["tokenized_text"])

tuning_rows = []
candidates = {}
for c_value in CFG.svm_c_values:
    candidate = LinearSVC(C=c_value, random_state=CFG.seed)
    candidate.fit(x_train, parts["train"]["label"])
    validation_prediction = candidate.predict(x_validation)
    tuning_rows.append({
        "C": c_value,
        **metric_row(parts["validation"]["label"], validation_prediction),
    })
    candidates[c_value] = candidate

tuning = (
    pd.DataFrame(tuning_rows)
    .sort_values(["f1_macro", "C"], ascending=[False, True])
    .reset_index(drop=True)
)
best_c = float(tuning.iloc[0]["C"])
model = candidates[best_c]
display(tuning)
print("Selected C using validation macro F1:", best_c)

x_test = vectorizer.transform(parts["test"]["tokenized_text"])
test_prediction = model.predict(x_test)
test_metrics = pd.DataFrame([{
    "model": "CLAWS word TF-IDF + Linear SVM",
    **metric_row(parts["test"]["label"], test_prediction),
}])
display(test_metrics)

tuning.to_csv(FINAL_OUT / "svm_validation_tuning.csv", index=False)
test_metrics.to_csv(FINAL_OUT / "final_test_metrics.csv", index=False)
joblib.dump(
    {
        "pipeline_version": CFG.pipeline_version,
        "labels": LABELS,
        "claws_version": claws_version,
        "vectorizer": vectorizer,
        "model": model,
        "best_c": best_c,
    },
    FINAL_OUT / "khmer_emotion_claws_tfidf_svm.joblib",
)


## 9. Final test evaluation and error analysis

This section reports the only final test evaluation. In addition to aggregate metrics,
it includes per-class performance, a normalized confusion matrix, group-aware bootstrap
uncertainty, influential TF–IDF features, nearest-training-text diagnostics, and every
test error.


In [ ]:
test_results = parts["test"][[
    "text_key", "text", "label", "near_duplicate_group", "template_group"
]].copy()
test_results["prediction"] = test_prediction
test_results["correct"] = test_results["label"].eq(test_results["prediction"])

report = pd.DataFrame(
    classification_report(
        test_results["label"],
        test_results["prediction"],
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )
).T
display(report)

matrix = confusion_matrix(
    test_results["label"],
    test_results["prediction"],
    labels=LABELS,
    normalize="true",
)
plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=LABELS,
    yticklabels=LABELS,
    vmin=0,
    vmax=1,
)
plt.title("Final template-family-disjoint test confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig(FINAL_OUT / "final_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

def grouped_bootstrap_macro_f1(frame, predictions, repeats, seed):
    rng = np.random.default_rng(seed)
    groups = frame["template_group"].drop_duplicates().to_numpy()
    group_positions = {
        group: np.flatnonzero(frame["template_group"].to_numpy() == group)
        for group in groups
    }
    truth = frame["label"].to_numpy()
    predictions = np.asarray(predictions)
    scores = []
    for _ in tqdm(range(repeats), desc="Group bootstrap"):
        sampled_groups = rng.choice(groups, size=len(groups), replace=True)
        positions = np.concatenate([group_positions[group] for group in sampled_groups])
        scores.append(
            f1_score(
                truth[positions],
                predictions[positions],
                labels=LABELS,
                average="macro",
                zero_division=0,
            )
        )
    return np.percentile(scores, [2.5, 50, 97.5])

macro_f1_interval = grouped_bootstrap_macro_f1(
    test_results,
    test_prediction,
    repeats=CFG.bootstrap_repeats,
    seed=CFG.seed,
)
print(
    "Macro-F1 group-bootstrap [2.5%, median, 97.5%]:",
    np.round(macro_f1_interval, 4),
)

feature_names = np.asarray(vectorizer.get_feature_names_out())
top_features = {}
for class_index, label in enumerate(model.classes_):
    strongest = np.argsort(model.coef_[class_index])[-12:][::-1]
    top_features[label] = feature_names[strongest]
top_features_table = pd.DataFrame(top_features)
display(top_features_table)

nearest_model = NearestNeighbors(
    n_neighbors=1, metric="cosine", algorithm="brute", n_jobs=-1
).fit(x_train)
distances, neighbor_indices = nearest_model.kneighbors(x_test)
similarities = 1 - distances[:, 0]
test_results["nearest_train_text"] = parts["train"].iloc[
    neighbor_indices[:, 0]
]["text"].to_numpy()
test_results["nearest_train_label"] = parts["train"].iloc[
    neighbor_indices[:, 0]
]["label"].to_numpy()
test_results["nearest_train_cosine"] = similarities

for threshold in [0.80, 0.90, 0.95]:
    print(
        f"Test texts with nearest-training cosine >= {threshold:.2f}: "
        f"{np.mean(similarities >= threshold):.2%}"
    )

errors = test_results.loc[~test_results["correct"]].sort_values(
    "nearest_train_cosine", ascending=False
)
print("Test errors:", len(errors), "of", len(test_results))
display(errors.head(30))

report.to_csv(FINAL_OUT / "final_classification_report.csv")
top_features_table.to_csv(FINAL_OUT / "top_features_by_class.csv", index=False)
test_results.to_parquet(FINAL_OUT / "final_test_predictions.parquet", index=False)
errors.to_csv(FINAL_OUT / "final_test_errors.csv", index=False)


## 10. Inference demonstration

The saved statistical pipeline expects Khmer text. It applies the same Unicode and
whitespace normalization, CLAWS segmentation, fitted TF–IDF transformation, and SVM
classifier used in the experiment. SVM decision values are ranking scores, not
calibrated probabilities.


In [ ]:
def predict_emotion(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input must be a non-empty string")
    normalized = normalize_base(text)
    if not KHMER_RE.search(normalized):
        raise ValueError("Input must contain Khmer characters")

    tokens = [token.strip() for token in segment(normalized) if token.strip()]
    features = vectorizer.transform([" ".join(tokens)])
    prediction = model.predict(features)[0]
    decision_scores = model.decision_function(features)[0]
    ranked_scores = {
        label: float(score)
        for label, score in sorted(
            zip(model.classes_, decision_scores),
            key=lambda item: item[1],
            reverse=True,
        )
    }
    return {
        "text": text,
        "normalized_text": normalized,
        "tokens": tokens,
        "predicted_emotion": prediction,
        "decision_scores": ranked_scores,
    }

demonstration_texts = [
    "ខ្ញុំសប្បាយចិត្តណាស់ដែលការងាររបស់ខ្ញុំបានជោគជ័យ។",
    "ខ្ញុំមានអារម្មណ៍ភ័យពេលដើរម្នាក់ឯងនៅពេលយប់។",
    "ការងារច្រើនពេកធ្វើឱ្យខ្ញុំមានសម្ពាធខ្លាំង។",
]
for demonstration_text in demonstration_texts:
    result = predict_emotion(demonstration_text)
    print(result["text"])
    print("Predicted emotion:", result["predicted_emotion"])
    print("Top decision scores:", list(result["decision_scores"].items())[:3])
    print()


## 11. Limitations and conclusion

### Limitations

- The source corpus is highly synthetic and template-heavy. Cleaning millions of rows
  leaves 287,284 exact-unique texts, 8,898 initial near-unique families, and 7,558
  retained benchmark examples.
- Strong performance on this benchmark may partly reflect class-specific vocabulary and
  generator writing patterns. It is not evidence of equal performance on natural Khmer
  social-media posts or conversations.
- Direct emotion-word masking is used only to improve template-family detection. It does
  not remove every possible semantic cue, and it is not part of model training.
- The class distribution is approximately, rather than perfectly, balanced. Macro F1 is
  therefore the primary metric.
- Labels are treated as mutually exclusive, although real emotional language can express
  multiple emotions simultaneously.
- This classifier is an educational NLP system, not a psychological or medical diagnostic
  tool.

### Conclusion

The project implements a complete Khmer statistical NLP pipeline: audited cleaning,
Khmer-specific tokenization, training-only TF–IDF fitting, validation-based SVM selection,
template-family-disjoint evaluation, error analysis, saved artifacts, and inference on new
Khmer text. The final test result should always be reported together with the synthetic-data
limitation.


## 12. Completion checklist

The final cell verifies that all submission artifacts were produced. Keep the executed
outputs when exporting or submitting the final notebook so the results can be inspected
without rerunning the expensive cleaning stages.


In [ ]:
required_artifacts = [
    FINAL_OUT / "final_template_group_split.parquet",
    FINAL_OUT / "svm_validation_tuning.csv",
    FINAL_OUT / "final_test_metrics.csv",
    FINAL_OUT / "final_classification_report.csv",
    FINAL_OUT / "final_confusion_matrix.png",
    FINAL_OUT / "final_test_predictions.parquet",
    FINAL_OUT / "final_test_errors.csv",
    FINAL_OUT / "khmer_emotion_claws_tfidf_svm.joblib",
]
missing_artifacts = [str(path) for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    raise RuntimeError(f"Missing final artifacts: {missing_artifacts}")

completion = {
    "benchmark_rows": int(len(sample)),
    "train_rows": int(len(parts["train"])),
    "validation_rows": int(len(parts["validation"])),
    "test_rows": int(len(parts["test"])),
    "template_families": int(sample["template_group"].nunique()),
    "best_c": best_c,
    "test_metrics": test_metrics.iloc[0].to_dict(),
    "macro_f1_bootstrap_95_percent": macro_f1_interval.tolist(),
    "model_path": str(FINAL_OUT / "khmer_emotion_claws_tfidf_svm.joblib"),
}
(FINAL_OUT / "completion_summary.json").write_text(
    json.dumps(completion, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(completion, ensure_ascii=False, indent=2))
print("Final submission pipeline completed successfully.")
